In [ ]:
# save as: find_links_where_proposal_wins_all_metrics.py
from __future__ import annotations

from pathlib import Path
from functools import reduce
import numpy as np
import pandas as pd


BASE_DIR = Path("results_sbrc_26") / "metrics"

FILES = {
    "GRU": BASE_DIR / "metrics_gru.csv",
    "LSTM": BASE_DIR / "metrics_lstm.csv",
    "PROPOSAL": BASE_DIR / "metrics_proposal.csv",
    "SARIMA": BASE_DIR / "metrics_sarima.csv",
}

# Métricas: menor é melhor / maior é melhor
LOWER_BETTER = {"dtw", "smape", "mae", "rmse", "nrmse_mean"}
HIGHER_BETTER = {"r2", "correncapa"}

# Tolerância para empates numéricos (evita falso "pior" por arredondamento)
EPS = 1e-12


def _read(name: str, path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"[{name}] arquivo não encontrado: {path}")
    df = pd.read_csv(path)
    required = {"link", "protocol"}
    if not required.issubset(df.columns):
        raise ValueError(f"[{name}] faltam colunas obrigatórias {required} em {path}")
    df["model"] = name
    return df


def _common_metrics(dfs: list[pd.DataFrame]) -> list[str]:
    # métricas em comum entre todos
    base = set(dfs[0].columns) - {"link", "protocol", "model"}
    for df in dfs[1:]:
        base &= (set(df.columns) - {"link", "protocol", "model"})
    # só as que sabemos comparar (para não quebrar se aparecer extra)
    keep = [m for m in sorted(base) if (m in LOWER_BETTER) or (m in HIGHER_BETTER)]
    if not keep:
        raise ValueError("Nenhuma métrica comparável em comum (esperava dtw, smape, mae, rmse, nrmse_mean, r2, correncapa).")
    return keep


def _merge_on_link_protocol(dfs: dict[str, pd.DataFrame]) -> pd.DataFrame:
    # renomeia colunas de métricas por sufixo do modelo
    renamed = []
    for name, df in dfs.items():
        cols = {c: f"{c}__{name}" for c in df.columns if c not in {"link", "protocol", "model"}}
        renamed.append(df.drop(columns=["model"]).rename(columns=cols))

    merged = reduce(lambda a, b: a.merge(b, on=["link", "protocol"], how="inner"), renamed)
    return merged


def _proposal_better_all(merged: pd.DataFrame, metrics: list[str]) -> pd.Series:
    ok = pd.Series(True, index=merged.index)

    # proposal < others for lower-better metrics; proposal > others for higher-better metrics
    others = [k for k in FILES.keys() if k != "PROPOSAL"]

    for m in metrics:
        pcol = f"{m}__PROPOSAL"
        # se faltar a coluna, ignora
        if pcol not in merged.columns:
            continue

        # NaN => falha (não conta como "melhor")
        ok &= np.isfinite(merged[pcol].values)

        for o in others:
            ocol = f"{m}__{o}"
            if ocol not in merged.columns:
                continue
            ok &= np.isfinite(merged[ocol].values)

            if m in LOWER_BETTER:
                ok &= (merged[pcol] <= merged[ocol] - EPS)
            else:  # higher better
                ok &= (merged[pcol] >= merged[ocol] + EPS)

    return ok


def main():
    dfs = {name: _read(name, path) for name, path in FILES.items()}
    metrics = _common_metrics(list(dfs.values()))
    merged = _merge_on_link_protocol(dfs)

    mask = _proposal_better_all(merged, metrics)
    winners = merged.loc[mask, ["link", "protocol"]].sort_values(["protocol", "link"]).reset_index(drop=True)

    # imprime
    print("Métricas comparadas:", metrics)
    print(f"\nLinks onde PROPOSAL é melhor em TODAS as métricas ({len(winners)}):\n")
    if len(winners) == 0:
        print("(nenhum)")
    else:
        print(winners.to_string(index=False))

    # salva lista
    out_path = BASE_DIR / "links_where_proposal_wins_all_metrics.csv"
    winners.to_csv(out_path, index=False, encoding="utf-8")
    print(f"\n[SALVO] {out_path}")

    # (Opcional) salvar também o merged filtrado, para auditoria
    audit_path = BASE_DIR / "audit_only_winners_full_metrics.csv"
    merged.loc[mask].to_csv(audit_path, index=False, encoding="utf-8")
    print(f"[SALVO] {audit_path}")


if __name__ == "__main__":
    main()
